In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from matplotlib.colors import LinearSegmentedColormap
from sklearn.svm import SVR
from sklearn.model_selection import KFold, cross_val_score

In [2]:
# 读取数据
df = pd.read_excel()
df.head()

# 编码
label_cols = ['Adsorbent', 'MOF type', 'Cation','Metal space group']
label_encoders = {}
for col in label_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

In [9]:
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from math import sqrt
import numpy as np

# 定义超参数搜索空间
param_grid = {
    'n_estimators': [300, 500, 700, 900],
    'learning_rate': [0.1, 0.2],
    'max_depth': [2, 3, 4],
    'subsample': [0.4, 0.6, 1.0],
    'colsample_bytree': [0.6, 0.8],
    'reg_alpha': [0, 0.5],
    'reg_lambda': [0, 1]
}

# 初始化模型
base_model = XGBRegressor(random_state=0)

# 嵌套交叉验证结构
# 内层：用于调参
inner_cv = KFold(n_splits=5, shuffle=True, random_state=1)
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='r2',  # 用 R² 作为评估指标
    cv=inner_cv,
    verbose=1
)

# 外层：用于评估泛化能力（替代独立测试集）
outer_cv = KFold(n_splits=5, shuffle=True, random_state=2)
nested_scores = cross_val_score(grid_search, X=x_train, y=y_train, cv=outer_cv, scoring='r2')

# 打印嵌套CV结果
print("嵌套交叉验证 R² 平均值: %.4f" % nested_scores.mean())
print("R² 区间: %.4f ~ %.4f" % (nested_scores.min(), nested_scores.max()))

# （可选）用最佳参数在整个训练集拟合最终模型并在测试集上评估
grid_search.fit(x_train, y_train)
best_params = grid_search.best_params_
print("最佳参数: ", best_params)

# 最终模型训练
final_model = XGBRegressor(**best_params, random_state=0)
final_model.fit(x_train, y_train)

# 在测试集上预测
y_pred = final_model.predict(x_test)

# 各项指标
print("训练集 R²: %.4f" % final_model.score(x_train, y_train))
print("测试集 R²: %.4f" % final_model.score(x_test, y_test))
print("MSE: %.4f" % mean_squared_error(y_test, y_pred))
print("RMSE: %.4f" % sqrt(mean_squared_error(y_test, y_pred)))
print("MAE: %.4f" % mean_absolute_error(y_test, y_pred))

# MAPE 定义
def mean_absolute_percentage_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("MAPE: %.4f%%" % mean_absolute_percentage_error(y_test, y_pred))

Fitting 5 folds for each of 576 candidates, totalling 2880 fits
Fitting 5 folds for each of 576 candidates, totalling 2880 fits
Fitting 5 folds for each of 576 candidates, totalling 2880 fits
Fitting 5 folds for each of 576 candidates, totalling 2880 fits
Fitting 5 folds for each of 576 candidates, totalling 2880 fits
嵌套交叉验证 R² 平均值: 0.9388
R² 区间: 0.8976 ~ 0.9656
Fitting 5 folds for each of 576 candidates, totalling 2880 fits
最佳参数:  {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 500, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.4}
训练集 R²: 0.9996
测试集 R²: 0.9659
MSE: 0.1811
RMSE: 0.4256
MAE: 0.3005
MAPE: 23.5512%
